### Notebook to run a closed-loop test of the Selector-PI controller (part of Biogoals.Select tool)

**Authors:** Davide Carecci  
**Created:** April 2025  
**Last Modified:** April 24, 2026  
**Version:** 1.1  
**Institution:** Politecnico di Milano  

---

#### Revision History

| Date       | Version | Author(s)              | Description                                                                                                              |
|------------|---------|------------------------|--------------------------------------------------------------------------------------------------------------------------|
| 2025-04-01 | 1.0     | D. Carecci             | Initial implementation for selector-PI design adjustment before the actual experiment done in Chile                      |
| 2026-04-24 | 1.1     | D. Carecci             | Code and folder cleaning and documentation for handle over a copy to A2A S.p.A                                           |

---
_Note_: this notebook was formalized starting from the one used to conduct the adjustment of the selector-PI design before the second experimental validation of the B.Select tool in Chile, as present in [Carecci *et al.*, 2026](http://ssrn.com/abstract=6024516). The original code structure for such sub-tool in also available: contact the authors to request a copy.
See also the B.Select user manual for more informations. 

In general, this code runs recursively "**test_closedloop_integrator.py**" (high-fidelity model present in the *Biogoals.agri-AcoDM* repository as real-plant emulator) and "**RPi_selectorPI_operative.py**" (where the actual selector-PI controller is implemented, exploiting the "**SeelctorPI_controller.py**" class).
After a complete test, it is possible to "post-process" the results.

### RUN THE CLOSED-LOOP 

In [ ]:
# Common standard python libraries that are needed
import numpy as np
import logging
import os
import pandas as pd
from datetime import datetime, timedelta
import ast
import sys
import time

In [ ]:
# # First-time setup guidance (run manually only if imports fail on a new machine)
# # Custom Python functions that are needed
# # 1) Save the current notebook working directory so you can return to it later
# original_dir = os.getcwd()
# # 2) Choose where to clone external repositories (EDIT this path)
# clone_dir = r"C:\path\to\my_external_libs"
# # 3) Move to that folder
# %cd {clone_dir}
# # 4) Clone the required repositories
# !git clone https://github.com/DaveCacci/B.Agri_AcoDM.git
# !git clone https://github.com/DaveCacci/B.Twin.git
# !git clone https://github.com/DaveCacci/general_utils.git
# # 5) Add cloned folders to Python import path
# sys.path.insert(0, rf"{clone_dir}\B.Agri_AcoDM") # clone_dir = path_to_agriacodm
# sys.path.insert(0, rf"{clone_dir}\B.Twin") # clone_dir = path_to_twin_library
# sys.path.insert(0, rf"{clone_dir}\general_utils") # clone_dir = path_to\general_utils
# # 6) Return to the original notebook working directory
# %cd {original_dir}
from test_closedloop_integrator import main as integrator_main # Uncomment only if you want to test a real closed-loop with the B.Agri_AcoDM as real plant simulator
from RPi_selectorPI_operative import main as main_selector

In [ ]:
# Define the number of closed-loop days you want to simulate
days_of_simulation = 45
control_interval = 6 # in hours (to be coherent with the values inside integrator.json)
control_steps = int(days_of_simulation*24/control_interval)
original_directory = os.getcwd()

# DECLARE LOGGER
# Configure logging
logger = logging.getLogger()
# Clean up any existing handlers for this logger
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
    handler.close()
# Console handler for displaying logs in the terminal
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_formatter = logging.Formatter('%(levelname)s - %(message)s')
console_handler.setFormatter(console_formatter)
# Add handlers to the logger
logger.addHandler(console_handler)
logger.setLevel(logging.INFO)

# When testing SelectorPI I must set the now time
current_timestamp = datetime(2025, 5, 30, 10) #datetime(2025, 5, 7, 13) #13:00 at index=0
control_steps = 2 # Set to 2 for testing purposes, but should be set to int(days_of_simulation*24/control_interval) for the actual simulation 
for i in range(control_steps-1):
    os.chdir(original_directory)

    integrator_main() # Uncomment only if you want to test a real closed-loop with the B.Agri_AcoDM as real plant simulator
    os.chdir(original_directory)
    
    # When testing SelectorPI I must set the now time
    main_selector('1', now = current_timestamp)
    current_timestamp += timedelta(hours=control_interval)   
    os.chdir(original_directory)

    # Sleep for the duration of the control interval
    print('-------------------------------------Sleeping for 1 seconds...-----------------------------------------')
    time.sleep(1)  # Sleep for the specified interval (in seconds)

### POST-PROCESS OF RESULTS
Add. See "**test_Chile_closedloop.ipynb**" in the '*Biogoals.Twin*' repository for a reference of the post-processing that can be done on the closed-loop results to generate meaningful plots etc.